# Graph models

Build a **transaction graph** from tabular feature importance, then train GraphSAGE, GAT, and SIGN-style GraphSAGE on the cleaned IEEE-CIS table.

1. Rank identity *pieces* from RF / LightGBM / XGBoost and stitch them into edge keys (`saved/graph_edge_schema.json`).
2. Train full-batch **GraphSAGE** and **GAT** on the undirected temporal k-NN union (baseline).
3. Train **SIGN GraphSAGE (v2)** on past-only identity edges, with an MCC threshold.

Holdout charts and comparisons with RF / LightGBM / XGBoost are in `03_graph_models_result.ipynb`.

Kernel: `graph` (PyTorch). Schema generation can fall back to `saved/graph_edge_schema.json` if LightGBM/XGBoost pickles are not in this env.


## Why a graph? (from the tabular models)

`saved/ml_results.parquet` is the **ablation table** (all RF / LightGBM / XGBoost runs), not per-column importances. Group-level signal is still clear:

| Group removed | Mean Δ ROC-AUC (baseline − removed) | Mean Δ PR-AUC | Interpretation |
|---|---|---|---|
| **C** (counts) | **+0.011** | **+0.040** | Only group that consistently *hurts* when dropped |
| M (match flags) | +0.003 | +0.002 | Mild |
| D (time/distance) | +0.002 | ~0 | Mild |
| id_* | −0.001 | ~0 | Safe to drop |
| **V\*** | **−0.006** | −0.004 | Removing V *improves* ROC-AUC |

The compact winner in `02_ml_models_result.ipynb` is **RF / LightGBM / XGBoost — Remove id + V** (~60 features, ROC-AUC ≈ 0.91).

A LightGBM gain fit on that set (`saved/lgb_column_importance.parquet`) ranks **card1, card2, TransactionAmt, TransactionDT, uid, addr1, C13, uid2, D15, D2, D1, C1** at the top. Consensus with RF and XGBoost (mean normalized importance) is written to `saved/model_column_importance.parquet`.

Those columns split cleanly:

- **Identity pieces** (`card*`, `addr*`, email, device, `uid`/`uid2`) → **edges**. Low-cardinality keys such as `ProductCD` are dropped as *single* keys (they would chain unrelated people). They can still appear inside a concatenated pair.
- **Behaviour** (`TransactionAmt`, `TransactionDT`, C counts, D delays) → **node features only**. Linking “similar amount” would just draw a k-NN on money, not an identity graph.

**Stitching.** Top identity pieces become:

1. one temporal k-NN graph per piece (`card1`, `addr1`, …), and
2. pair keys `link_A_B` = `A + "_" + B` for the strongest pairs (the same construction as `uid` = `card1_card2_card3_card5`).

`uid` and `uid2` stay in the union so the old identity graph is a subset, not a replacement.

**Hypothesis.** Fraud reuses *parts* of an identity (same card, same email+addr, same device). A GNN that aggregates along those pieces — and along the concatenated keys — can see sharing patterns that a single `uid`/`uid2` chain misses.

## 1. Graph from tabular feature importance

Identity pieces (`card*`, `addr*`, email, device, `uid` / `uid2`) become **edge keys**. C counts, D delays, amount, and time stay as **node features**. Complementary pairs are concatenated the same way `uid = card1_card2_card3_card5` was.


In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.display.precision = 4

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

ROOT = Path("/content/drive/MyDrive/minor-thesis") if IS_COLAB else Path.cwd()
DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
print(f"ROOT={ROOT}")

ROOT=d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis


## Helper functions (edge keys and temporal k-NN)


In [2]:
"""Turn RF / LightGBM / XGBoost feature importance into graph edge keys.

Identity *pieces* (card, addr, email, device, uid) become:
- single-key temporal edges, and
- concatenated pair keys ("string the pieces together").

Numeric behaviour columns (C, D, amount, time) stay as *node features*.
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd

MODEL_DIR = SAVED_PATH / "ml_test_models"
SCHEMA_PATH = SAVED_PATH / "graph_edge_schema.json"

IDENTITY_PIECES = {
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceType",
    "DeviceInfo",
    "ProductCD",
    "uid",
    "uid2",
}

MISSING_VALUES = {-999, -1}
MIN_UNIQUES_SINGLE = 40
MAX_GROUP_FRAC = 0.05
TOP_PIECES = 8
TOP_PAIRS = 10
K_NEIGHBORS = 5

# uid = card1_card2_card3_card5; uid2 = uid_addr1_P_emaildomain.
# Pairing a composite with one of its own pieces just rebuilds the same chain.
UID_CONTAINED = {"card1", "card2", "card3", "card5"}
UID2_CONTAINED = UID_CONTAINED | {"uid", "addr1", "P_emaildomain"}


def _redundant_pair(a: str, b: str) -> bool:
    pair = {a, b}
    if "uid" in pair and pair & UID_CONTAINED:
        return True
    if "uid2" in pair and pair & UID2_CONTAINED:
        return True
    return False


def _latest(pattern: str) -> Path:
    hits = sorted(MODEL_DIR.glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No files matching {MODEL_DIR / pattern}")
    return hits[-1]


def load_model_importances() -> pd.DataFrame:
    import joblib

    feat_path = _latest("features_*.json")
    with open(feat_path, encoding="utf-8") as f:
        names = json.load(f)["feature_names"]

    models = {
        "RandomForest": joblib.load(_latest("randomforest_*.pkl")),
        "LightGBM": joblib.load(_latest("lightgbm_*.pkl")),
        "XGBoost": joblib.load(_latest("xgboost_*.pkl")),
    }
    rows = []
    for model_name, model in models.items():
        imp = np.asarray(model.feature_importances_, dtype=np.float64)
        if imp.sum() > 0:
            imp = imp / imp.sum()
        for feat, val in zip(names, imp):
            rows.append({"model": model_name, "feature": feat, "importance": float(val)})
    long = pd.DataFrame(rows)
    wide = long.pivot(index="feature", columns="model", values="importance").fillna(0.0)
    wide["mean"] = wide.mean(axis=1)
    wide["rank"] = wide["mean"].rank(ascending=False, method="min").astype(int)
    wide["is_identity_piece"] = wide.index.isin(IDENTITY_PIECES)
    return wide.sort_values("mean", ascending=False).reset_index()


def _nunique(series: pd.Series) -> int:
    return int(series.nunique(dropna=True))


def propose_edge_keys(df: pd.DataFrame, importance: pd.DataFrame) -> list[dict]:
    pieces = (
        importance.loc[importance["is_identity_piece"]]
        .sort_values("mean", ascending=False)["feature"]
        .tolist()
    )
    pieces = [p for p in pieces if p in df.columns][:TOP_PIECES]
    keys: list[dict] = []

    def add_key(name: str, parts: list[str], source: str, score: float) -> None:
        if any(p not in df.columns for p in parts):
            return
        if any(k["name"] == name for k in keys):
            return
        keys.append(
            {
                "name": name,
                "parts": parts,
                "source": source,
                "score": float(score),
            }
        )

    score_map = dict(zip(importance["feature"], importance["mean"]))

    for piece in pieces:
        nunq = _nunique(df[piece])
        if nunq < MIN_UNIQUES_SINGLE:
            continue
        add_key(piece, [piece], "single", score_map.get(piece, 0.0))

    pair_cands = []
    for i, a in enumerate(pieces):
        for b in pieces[i + 1 :]:
            if _redundant_pair(a, b):
                continue
            pair_cands.append((score_map.get(a, 0.0) + score_map.get(b, 0.0), a, b))
    pair_cands.sort(reverse=True)
    for score, a, b in pair_cands[:TOP_PAIRS]:
        add_key(f"link_{a}_{b}", [a, b], "pair", score)

    # Always keep the original engineered identities if present.
    for extra in ("uid", "uid2"):
        if extra in df.columns:
            add_key(extra, [extra], "engineered", score_map.get(extra, 0.0))

    keys.sort(key=lambda k: -k["score"])
    return keys


def key_series(df: pd.DataFrame, parts: list[str]) -> pd.Series:
    if len(parts) == 1:
        return df[parts[0]]
    cols = [df[p].to_numpy() for p in parts]
    return pd.Series(list(zip(*cols)), index=df.index)


def invalid_group_mask(df: pd.DataFrame, parts: list[str]) -> pd.Series:
    mask = pd.Series(False, index=df.index)
    for p in parts:
        mask |= df[p].isin(MISSING_VALUES)
    return mask


def build_temporal_knn_for_key(df: pd.DataFrame, parts: list[str], k: int = K_NEIGHBORS) -> np.ndarray:
    keys = key_series(df, parts)
    invalid = invalid_group_mask(df, parts)
    src_parts: list[np.ndarray] = []
    dst_parts: list[np.ndarray] = []
    max_group = int(len(df) * MAX_GROUP_FRAC)

    work = df.loc[~invalid, ["TransactionDT"]].copy()
    work["_k"] = keys.loc[~invalid].to_numpy()
    for _, group in work.groupby("_k", sort=False):
        n = len(group)
        if n < 2 or n > max_group:
            continue
        idx = group.sort_values("TransactionDT").index.to_numpy(dtype=np.int64, copy=False)
        for offset in range(1, min(k + 1, n)):
            left, right = idx[:-offset], idx[offset:]
            src_parts.extend((left, right))
            dst_parts.extend((right, left))
    if not src_parts:
        return np.zeros((2, 0), dtype=np.int64)
    return np.vstack([np.concatenate(src_parts), np.concatenate(dst_parts)])


def coalesce_edges(edge_index: np.ndarray, num_nodes: int) -> np.ndarray:
    if edge_index.size == 0:
        return edge_index
    packed = edge_index[0].astype(np.int64) * np.int64(num_nodes) + edge_index[1].astype(np.int64)
    _, uniq = np.unique(packed, return_index=True)
    return edge_index[:, np.sort(uniq)]


def build_union_edges(df: pd.DataFrame, edge_keys: list[dict], k: int = K_NEIGHBORS) -> tuple[np.ndarray, list[dict]]:
    stats = []
    chunks = []
    for spec in edge_keys:
        edges = build_temporal_knn_for_key(df, spec["parts"], k=k)
        stats.append(
            {
                **spec,
                "directed_edges": int(edges.shape[1]),
            }
        )
        print(
            f"  {spec['name']:30s}  {spec['source']:10s}  "
            f"parts={'+'.join(spec['parts']):40s}  edges={edges.shape[1]:,}",
            flush=True,
        )
        if edges.shape[1]:
            chunks.append(edges)
    if not chunks:
        union = np.zeros((2, 0), dtype=np.int64)
    else:
        union = coalesce_edges(np.concatenate(chunks, axis=1), len(df))
    return union, stats


def load_schema(path: Path = SCHEMA_PATH) -> dict:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def resolve_edge_keys(df: pd.DataFrame) -> list[dict]:
    """Prefer a saved schema so the GNN venv does not need LightGBM/XGBoost."""
    if SCHEMA_PATH.exists():
        schema = load_schema()
        return schema["edge_keys"]
    try:
        importance = load_model_importances()
        edge_keys = propose_edge_keys(df, importance)
        save_schema(importance, edge_keys)
        return edge_keys
    except Exception as exc:
        print(f"Could not load model importances ({exc}); falling back to uid/uid2")
        return [
            {"name": "uid", "parts": ["uid"], "source": "engineered", "score": 1.0},
            {"name": "uid2", "parts": ["uid2"], "source": "engineered", "score": 1.0},
        ]


def save_schema(importance: pd.DataFrame, edge_keys: list[dict], path: Path = SCHEMA_PATH) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "pieces": importance.loc[importance["is_identity_piece"], "feature"].head(TOP_PIECES).tolist(),
        "edge_keys": edge_keys,
        "rules": {
            "min_uniques_single": MIN_UNIQUES_SINGLE,
            "max_group_frac": MAX_GROUP_FRAC,
            "k_neighbors": K_NEIGHBORS,
            "top_pieces": TOP_PIECES,
            "top_pairs": TOP_PAIRS,
        },
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    importance.to_parquet(SAVED_PATH / "model_column_importance.parquet", index=False)
    return path




def to_csr(edge_index, num_nodes):
    if edge_index.size == 0:
        indptr = np.zeros(num_nodes + 1, dtype=np.int64)
        indices = np.zeros(0, dtype=np.int64)
        return indptr, indices
    order = np.argsort(edge_index[0], kind="mergesort")
    src = edge_index[0, order]
    dst = edge_index[1, order]
    counts = np.bincount(src, minlength=num_nodes)
    indptr = np.zeros(num_nodes + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])
    return indptr, dst.astype(np.int64)


## 1.1 Consensus importance (RF + LightGBM + XGBoost)

Each model's `feature_importances_` is normalized to sum to 1, then averaged. Identity pieces become candidate edge keys; C / D / amount / time stay on the node.

In [3]:
importance = load_model_importances()
display(
    importance.head(20)[
        ["feature", "RandomForest", "LightGBM", "XGBoost", "mean", "rank", "is_identity_piece"]
    ]
)
print("Identity pieces")
display(importance.loc[importance["is_identity_piece"], ["feature", "mean", "rank"]])


model,feature,RandomForest,LightGBM,XGBoost,mean,rank,is_identity_piece
0,C8,0.0234,0.0055,0.1222,0.0504,1,False
1,C4,0.0200,0.0033,0.1134,0.0455,2,False
2,C14,0.0330,0.0220,0.0669,0.0406,3,False
3,C5,0.0329,0.0130,0.0565,0.0341,4,False
4,card1,0.0329,0.0613,0.0071,0.0338,5,True
5,TransactionAmt,0.0419,0.0490,0.0096,0.0335,6,False
6,uid,0.0331,0.0508,0.0093,0.0311,7,True
7,card2,0.0298,0.0545,0.0082,0.0308,8,True
8,addr1,0.0291,0.0540,0.0076,0.0302,9,True
9,C13,0.0395,0.0384,0.0116,0.0298,10,False


Identity pieces


model,feature,mean,rank
4,card1,0.0338,5
6,uid,0.0311,7
7,card2,0.0308,8
8,addr1,0.0302,9
15,card6,0.0219,16
16,R_emaildomain,0.0217,17
19,uid2,0.0203,20
22,P_emaildomain,0.0162,23
23,card5,0.0156,24
24,ProductCD,0.0150,25


## 1.2 Stitch pieces into edge keys

Singles (enough distinct values) plus the strongest non-redundant pairs. `uid` already contains `card1/2/3/5`, so pairs like `card1+uid` are skipped.

In [4]:
train = pd.read_parquet(DATASET_PATH / "merged_train.parquet")
train = train.sort_values("TransactionDT").reset_index(drop=True)
edge_keys = propose_edge_keys(train, importance)
save_schema(importance, edge_keys)
print(f"Wrote {SCHEMA_PATH}")
display(pd.DataFrame(edge_keys))

Wrote d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\graph_edge_schema.json


,name,parts,source,score
0,link_card1_card2,"[card1, card2]",pair,0.0646
1,link_card1_addr1,"[card1, addr1]",pair,0.0640
2,link_uid_addr1,"[uid, addr1]",pair,0.0613
3,link_card2_addr1,"[card2, addr1]",pair,0.0611
4,link_card1_card6,"[card1, card6]",pair,0.0557
5,link_card1_R_emaildomain,"[card1, R_emaildomain]",pair,0.0555
6,link_uid_card6,"[uid, card6]",pair,0.0530
7,link_card2_card6,"[card2, card6]",pair,0.0528
8,link_uid_R_emaildomain,"[uid, R_emaildomain]",pair,0.0528
9,link_card2_R_emaildomain,"[card2, R_emaildomain]",pair,0.0525


## 1.3 Union of temporal k-NN graphs (K = 5)

Inside each key group, sort by `TransactionDT` and link each transaction to its five nearest neighbors. Skip missing sentinels (`-999`, `-1`) and groups larger than 5% of the data. Deduplicate the union.

In [5]:
edge_index, edge_stats = build_union_edges(train, edge_keys, k=5)
indptr, _ = to_csr(edge_index, len(train))
deg = np.diff(indptr)
stats = pd.DataFrame(edge_stats)
display(stats[["name", "source", "directed_edges"]])
print(f"union unique directed edges: {edge_index.shape[1]:,}")
print(f"average degree: {deg.mean():.2f}  max: {deg.max()}  isolated: {(deg == 0).sum():,}")


  link_card1_card2                pair        parts=card1+card2                               edges=5,508,336
  link_card1_addr1                pair        parts=card1+addr1                               edges=4,506,446
  link_uid_addr1                  pair        parts=uid+addr1                                 edges=4,492,804
  link_card2_addr1                pair        parts=card2+addr1                               edges=4,990,332
  link_card1_card6                pair        parts=card1+card6                               edges=5,584,586
  link_card1_R_emaildomain        pair        parts=card1+R_emaildomain                       edges=5,357,210
  link_uid_card6                  pair        parts=uid+card6                                 edges=5,593,228
  link_card2_card6                pair        parts=card2+card6                               edges=4,244,072
  link_uid_R_emaildomain          pair        parts=uid+R_emaildomain                         edges=5,356,924
  link_car

,name,source,directed_edges
0,link_card1_card2,pair,5508336
1,link_card1_addr1,pair,4506446
2,link_uid_addr1,pair,4492804
3,link_card2_addr1,pair,4990332
4,link_card1_card6,pair,5584586
5,link_card1_R_emaildomain,pair,5357210
6,link_uid_card6,pair,5593228
7,link_card2_card6,pair,4244072
8,link_uid_R_emaildomain,pair,5356924
9,link_card2_R_emaildomain,pair,4214138


union unique directed edges: 21,451,920
average degree: 36.33  max: 116  isolated: 4


## 2. Train GraphSAGE and GAT (baseline)

Full-batch message passing on the stitched undirected k-NN graph (K = 5). GAT attention on ~21M edges may run out of CPU RAM; GraphSAGE still writes weights under `saved/graph_models/`.

This is the 0.767 ROC-AUC / ~46k-FP baseline. Section 3 is the improved trainer.


In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.display.precision = 4

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    ROOT = Path("/content/drive/MyDrive/minor-thesis")
else:
    ROOT = Path.cwd()

DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
RESULTS_DIR = SAVED_PATH / "graph_models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset: {DATASET_PATH}")
print(f"Results: {RESULTS_DIR}")

Running on Local
Dataset: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\dataset
Results: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\graph_models


## Graph helpers, GraphSAGE, and GAT


In [ ]:
"""Turn RF / LightGBM / XGBoost feature importance into graph edge keys.

Identity *pieces* (card, addr, email, device, uid) become:
- single-key temporal edges, and
- concatenated pair keys ("string the pieces together").

Numeric behaviour columns (C, D, amount, time) stay as *node features*.
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd

MODEL_DIR = SAVED_PATH / "ml_test_models"
SCHEMA_PATH = SAVED_PATH / "graph_edge_schema.json"

IDENTITY_PIECES = {
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceType",
    "DeviceInfo",
    "ProductCD",
    "uid",
    "uid2",
}

MISSING_VALUES = {-999, -1}
MIN_UNIQUES_SINGLE = 40
MAX_GROUP_FRAC = 0.05
TOP_PIECES = 8
TOP_PAIRS = 10
K_NEIGHBORS = 5

# uid = card1_card2_card3_card5; uid2 = uid_addr1_P_emaildomain.
# Pairing a composite with one of its own pieces just rebuilds the same chain.
UID_CONTAINED = {"card1", "card2", "card3", "card5"}
UID2_CONTAINED = UID_CONTAINED | {"uid", "addr1", "P_emaildomain"}


def _redundant_pair(a: str, b: str) -> bool:
    pair = {a, b}
    if "uid" in pair and pair & UID_CONTAINED:
        return True
    if "uid2" in pair and pair & UID2_CONTAINED:
        return True
    return False


def _latest(pattern: str) -> Path:
    hits = sorted(MODEL_DIR.glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No files matching {MODEL_DIR / pattern}")
    return hits[-1]


def load_model_importances() -> pd.DataFrame:
    import joblib

    feat_path = _latest("features_*.json")
    with open(feat_path, encoding="utf-8") as f:
        names = json.load(f)["feature_names"]

    models = {
        "RandomForest": joblib.load(_latest("randomforest_*.pkl")),
        "LightGBM": joblib.load(_latest("lightgbm_*.pkl")),
        "XGBoost": joblib.load(_latest("xgboost_*.pkl")),
    }
    rows = []
    for model_name, model in models.items():
        imp = np.asarray(model.feature_importances_, dtype=np.float64)
        if imp.sum() > 0:
            imp = imp / imp.sum()
        for feat, val in zip(names, imp):
            rows.append({"model": model_name, "feature": feat, "importance": float(val)})
    long = pd.DataFrame(rows)
    wide = long.pivot(index="feature", columns="model", values="importance").fillna(0.0)
    wide["mean"] = wide.mean(axis=1)
    wide["rank"] = wide["mean"].rank(ascending=False, method="min").astype(int)
    wide["is_identity_piece"] = wide.index.isin(IDENTITY_PIECES)
    return wide.sort_values("mean", ascending=False).reset_index()


def _nunique(series: pd.Series) -> int:
    return int(series.nunique(dropna=True))


def propose_edge_keys(df: pd.DataFrame, importance: pd.DataFrame) -> list[dict]:
    pieces = (
        importance.loc[importance["is_identity_piece"]]
        .sort_values("mean", ascending=False)["feature"]
        .tolist()
    )
    pieces = [p for p in pieces if p in df.columns][:TOP_PIECES]
    keys: list[dict] = []

    def add_key(name: str, parts: list[str], source: str, score: float) -> None:
        if any(p not in df.columns for p in parts):
            return
        if any(k["name"] == name for k in keys):
            return
        keys.append(
            {
                "name": name,
                "parts": parts,
                "source": source,
                "score": float(score),
            }
        )

    score_map = dict(zip(importance["feature"], importance["mean"]))

    for piece in pieces:
        nunq = _nunique(df[piece])
        if nunq < MIN_UNIQUES_SINGLE:
            continue
        add_key(piece, [piece], "single", score_map.get(piece, 0.0))

    pair_cands = []
    for i, a in enumerate(pieces):
        for b in pieces[i + 1 :]:
            if _redundant_pair(a, b):
                continue
            pair_cands.append((score_map.get(a, 0.0) + score_map.get(b, 0.0), a, b))
    pair_cands.sort(reverse=True)
    for score, a, b in pair_cands[:TOP_PAIRS]:
        add_key(f"link_{a}_{b}", [a, b], "pair", score)

    # Always keep the original engineered identities if present.
    for extra in ("uid", "uid2"):
        if extra in df.columns:
            add_key(extra, [extra], "engineered", score_map.get(extra, 0.0))

    keys.sort(key=lambda k: -k["score"])
    return keys


def key_series(df: pd.DataFrame, parts: list[str]) -> pd.Series:
    if len(parts) == 1:
        return df[parts[0]]
    cols = [df[p].to_numpy() for p in parts]
    return pd.Series(list(zip(*cols)), index=df.index)


def invalid_group_mask(df: pd.DataFrame, parts: list[str]) -> pd.Series:
    mask = pd.Series(False, index=df.index)
    for p in parts:
        mask |= df[p].isin(MISSING_VALUES)
    return mask


def build_temporal_knn_for_key(df: pd.DataFrame, parts: list[str], k: int = K_NEIGHBORS) -> np.ndarray:
    keys = key_series(df, parts)
    invalid = invalid_group_mask(df, parts)
    src_parts: list[np.ndarray] = []
    dst_parts: list[np.ndarray] = []
    max_group = int(len(df) * MAX_GROUP_FRAC)

    work = df.loc[~invalid, ["TransactionDT"]].copy()
    work["_k"] = keys.loc[~invalid].to_numpy()
    for _, group in work.groupby("_k", sort=False):
        n = len(group)
        if n < 2 or n > max_group:
            continue
        idx = group.sort_values("TransactionDT").index.to_numpy(dtype=np.int64, copy=False)
        for offset in range(1, min(k + 1, n)):
            left, right = idx[:-offset], idx[offset:]
            src_parts.extend((left, right))
            dst_parts.extend((right, left))
    if not src_parts:
        return np.zeros((2, 0), dtype=np.int64)
    return np.vstack([np.concatenate(src_parts), np.concatenate(dst_parts)])


def coalesce_edges(edge_index: np.ndarray, num_nodes: int) -> np.ndarray:
    if edge_index.size == 0:
        return edge_index
    packed = edge_index[0].astype(np.int64) * np.int64(num_nodes) + edge_index[1].astype(np.int64)
    _, uniq = np.unique(packed, return_index=True)
    return edge_index[:, np.sort(uniq)]


def build_union_edges(df: pd.DataFrame, edge_keys: list[dict], k: int = K_NEIGHBORS) -> tuple[np.ndarray, list[dict]]:
    stats = []
    chunks = []
    for spec in edge_keys:
        edges = build_temporal_knn_for_key(df, spec["parts"], k=k)
        stats.append(
            {
                **spec,
                "directed_edges": int(edges.shape[1]),
            }
        )
        print(
            f"  {spec['name']:30s}  {spec['source']:10s}  "
            f"parts={'+'.join(spec['parts']):40s}  edges={edges.shape[1]:,}",
            flush=True,
        )
        if edges.shape[1]:
            chunks.append(edges)
    if not chunks:
        union = np.zeros((2, 0), dtype=np.int64)
    else:
        union = coalesce_edges(np.concatenate(chunks, axis=1), len(df))
    return union, stats


def load_schema(path: Path = SCHEMA_PATH) -> dict:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def resolve_edge_keys(df: pd.DataFrame) -> list[dict]:
    """Prefer a saved schema so the GNN venv does not need LightGBM/XGBoost."""
    if SCHEMA_PATH.exists():
        schema = load_schema()
        return schema["edge_keys"]
    try:
        importance = load_model_importances()
        edge_keys = propose_edge_keys(df, importance)
        save_schema(importance, edge_keys)
        return edge_keys
    except Exception as exc:
        print(f"Could not load model importances ({exc}); falling back to uid/uid2")
        return [
            {"name": "uid", "parts": ["uid"], "source": "engineered", "score": 1.0},
            {"name": "uid2", "parts": ["uid2"], "source": "engineered", "score": 1.0},
        ]


def save_schema(importance: pd.DataFrame, edge_keys: list[dict], path: Path = SCHEMA_PATH) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "pieces": importance.loc[importance["is_identity_piece"], "feature"].head(TOP_PIECES).tolist(),
        "edge_keys": edge_keys,
        "rules": {
            "min_uniques_single": MIN_UNIQUES_SINGLE,
            "max_group_frac": MAX_GROUP_FRAC,
            "k_neighbors": K_NEIGHBORS,
            "top_pieces": TOP_PIECES,
            "top_pairs": TOP_PAIRS,
        },
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    importance.to_parquet(SAVED_PATH / "model_column_importance.parquet", index=False)
    return path




def to_csr(edge_index, num_nodes):
    if edge_index.size == 0:
        indptr = np.zeros(num_nodes + 1, dtype=np.int64)
        indices = np.zeros(0, dtype=np.int64)
        return indptr, indices
    order = np.argsort(edge_index[0], kind="mergesort")
    src = edge_index[0, order]
    dst = edge_index[1, order]
    counts = np.bincount(src, minlength=num_nodes)
    indptr = np.zeros(num_nodes + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])
    return indptr, dst.astype(np.int64)

"""Train GraphSAGE and GAT on IEEE-CIS fraud transactions.

Graph construction
------------------
Nodes are transactions. Node features match the compact "Remove id + V"
tabular set. Edges are a union of temporal k-NN graphs, one per identity key.

Keys are chosen from RF / LightGBM / XGBoost feature importance:
identity *pieces* (card, addr, email, device, uid) become single-key edges
and concatenated pair keys. C/D/amount/time stay as node features only.

Fallback if the importance schema is missing: uid and uid2 only.
"""

import gc
import json
import random
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from torch import nn

warnings.filterwarnings("ignore")

RESULTS_DIR = SAVED_PATH / "graph_models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
K_NEIGHBORS = 5
HIDDEN = 64
DROPOUT = 0.3
EPOCHS_SAGE = 25
EPOCHS_GAT = 15
PATIENCE = 6
BATCH_SIZE = 2048
NUM_NEIGHBORS = 10
LR = 1e-3
WEIGHT_DECAY = 5e-4


def set_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def select_node_features(columns: list[str]) -> list[str]:
    """Compact set aligned with 02_ml_models.ipynb (drop id_* and V*)."""
    cols = [
        c
        for c in columns
        if c not in {"isFraud", "TransactionID", "uid", "uid2"}
        and not c.startswith("id")
        and not c.startswith("V")
    ]
    # Keep identity keys as features as well; they are also used for edges.
    for extra in ("uid", "uid2"):
        if extra in columns and extra not in cols:
            cols.append(extra)
    return cols


def build_temporal_knn_edges(df: pd.DataFrame, key: str, k: int = K_NEIGHBORS) -> np.ndarray:
    """Connect each txn to up to k previous/next txns with the same identity key."""
    src_parts: list[np.ndarray] = []
    dst_parts: list[np.ndarray] = []
    for _, group in df.groupby(key, sort=False):
        n = len(group)
        if n < 2:
            continue
        idx = group.sort_values("TransactionDT").index.to_numpy(dtype=np.int64, copy=False)
        for offset in range(1, min(k + 1, n)):
            left, right = idx[:-offset], idx[offset:]
            src_parts.extend((left, right))
            dst_parts.extend((right, left))
    if not src_parts:
        return np.zeros((2, 0), dtype=np.int64)
    src = np.concatenate(src_parts)
    dst = np.concatenate(dst_parts)
    return np.vstack([src, dst])


def coalesce_edges(edge_index: np.ndarray, num_nodes: int) -> np.ndarray:
    if edge_index.size == 0:
        return edge_index
    key = edge_index[0].astype(np.int64) * np.int64(num_nodes) + edge_index[1].astype(np.int64)
    _, uniq = np.unique(key, return_index=True)
    return edge_index[:, np.sort(uniq)]


def to_csr(edge_index: np.ndarray, num_nodes: int):
    if edge_index.size == 0:
        indptr = np.zeros(num_nodes + 1, dtype=np.int64)
        indices = np.zeros(0, dtype=np.int64)
        return indptr, indices
    order = np.argsort(edge_index[0], kind="mergesort")
    src = edge_index[0, order]
    dst = edge_index[1, order]
    counts = np.bincount(src, minlength=num_nodes)
    indptr = np.zeros(num_nodes + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])
    return indptr, dst.astype(np.int64)


def sample_subgraph(seeds: np.ndarray, indptr: np.ndarray, indices: np.ndarray, num_neighbors: int, num_hops: int = 2):
    """Neighbor sampling used by GraphSAGE/GAT mini-batches."""
    rng = np.random.default_rng()
    node_to_local = {int(n): i for i, n in enumerate(seeds)}
    nodes = [int(n) for n in seeds]
    hop_nodes = list(seeds)
    for _ in range(num_hops):
        nxt = []
        for n in hop_nodes:
            start, end = int(indptr[n]), int(indptr[n + 1])
            neigh = indices[start:end]
            if neigh.size == 0:
                continue
            if neigh.size > num_neighbors:
                chosen = rng.choice(neigh, size=num_neighbors, replace=False)
            else:
                chosen = neigh
            for m in chosen:
                m = int(m)
                if m not in node_to_local:
                    node_to_local[m] = len(nodes)
                    nodes.append(m)
                nxt.append(m)
        hop_nodes = nxt
    nodes_arr = np.asarray(nodes, dtype=np.int64)
    src, dst = [], []
    for n in nodes_arr:
        local_n = node_to_local[int(n)]
        start, end = int(indptr[n]), int(indptr[n + 1])
        for m in indices[start:end]:
            m = int(m)
            if m in node_to_local:
                src.append(local_n)
                dst.append(node_to_local[m])
    if not src:
        eidx = torch.zeros((2, 0), dtype=torch.long)
    else:
        eidx = torch.tensor([src, dst], dtype=torch.long)
    return nodes_arr, eidx


def scatter_sum(src: torch.Tensor, index: torch.Tensor, dim_size: int) -> torch.Tensor:
    out = torch.zeros(dim_size, *src.shape[1:], device=src.device, dtype=src.dtype)
    out.index_add_(0, index, src)
    return out


def scatter_softmax(src: torch.Tensor, index: torch.Tensor, dim_size: int) -> torch.Tensor:
    """Numerically stable softmax over incoming edges of each destination node."""
    idx = index.view(-1, *([1] * (src.dim() - 1))).expand_as(src)
    maxes = torch.full((dim_size,) + src.shape[1:], torch.finfo(src.dtype).min, device=src.device, dtype=src.dtype)
    maxes.scatter_reduce_(0, idx, src, reduce="amax", include_self=True)
    exp = (src - maxes.index_select(0, index)).exp()
    denom = scatter_sum(exp, index, dim_size).clamp_min(1e-16)
    return exp / denom.index_select(0, index)


class SAGEConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.lin_self = nn.Linear(in_channels, out_channels)
        self.lin_neigh = nn.Linear(in_channels, out_channels)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        if edge_index.numel() == 0:
            return self.lin_self(x)
        row, col = edge_index[0], edge_index[1]
        neigh = scatter_sum(x[col], row, x.size(0))
        deg = scatter_sum(torch.ones(col.size(0), 1, device=x.device, dtype=x.dtype), row, x.size(0)).clamp_min(1.0)
        neigh = neigh / deg
        return self.lin_self(x) + self.lin_neigh(neigh)


class GATConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, heads: int = 4, dropout: float = 0.1, concat: bool = True):
        super().__init__()
        self.heads = heads
        self.out_channels = out_channels
        self.concat = concat
        self.dropout = dropout
        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.att_src = nn.Parameter(torch.empty(1, heads, out_channels))
        self.att_dst = nn.Parameter(torch.empty(1, heads, out_channels))
        self.bias = nn.Parameter(torch.zeros(heads * out_channels if concat else out_channels))
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        n, h, c = x.size(0), self.heads, self.out_channels
        xh = self.lin(x).view(n, h, c)
        if edge_index.numel() == 0:
            out = xh.reshape(n, h * c) if self.concat else xh.mean(dim=1)
            return out + self.bias
        row, col = edge_index[0], edge_index[1]
        alpha = (xh[row] * self.att_src).sum(-1) + (xh[col] * self.att_dst).sum(-1)
        alpha = F.leaky_relu(alpha, 0.2)
        alpha = scatter_softmax(alpha, row, n)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        msg = xh[col] * alpha.unsqueeze(-1)
        out = scatter_sum(msg, row, n)
        if self.concat:
            out = out.reshape(n, h * c)
        else:
            out = out.mean(dim=1)
        return out + self.bias


class GraphSAGE(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int = HIDDEN, dropout: float = DROPOUT):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.dropout = dropout
        self.classifier = nn.Linear(hidden_channels, 1)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x).squeeze(-1)


class GATNet(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int = 32, heads: int = 2, dropout: float = DROPOUT):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout, concat=True)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, dropout=dropout, concat=False)
        self.dropout = dropout
        self.classifier = nn.Linear(hidden_channels, 1)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x).squeeze(-1)


def train_one_model(
    name: str,
    model: nn.Module,
    x_all: torch.Tensor,
    y_all: np.ndarray,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    edge_index: torch.Tensor,
    pos_weight: torch.Tensor,
    device: torch.device,
    epochs: int,
):
    """One full-graph forward/backward per epoch (fits 590k nodes × ~6M edges)."""
    model = model.to(device)
    x = x_all.to(device)
    eidx = edge_index.to(device)
    y_t = torch.from_numpy(y_all.astype(np.float32)).to(device)
    train_t = torch.from_numpy(train_idx.astype(np.int64)).to(device)
    val_t = torch.from_numpy(val_idx.astype(np.int64)).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    history = []
    best_auc = -1.0
    best_state = None
    best_epoch = 0
    stalled = 0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, eidx)
        loss = criterion(logits[train_t], y_t[train_t])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x, eidx)[val_t]
            y_prob = torch.sigmoid(val_logits).cpu().numpy()
        y_true = y_all[val_idx]
        auc = roc_auc_score(y_true, y_prob)
        prauc = average_precision_score(y_true, y_prob)
        avg_loss = float(loss.item())
        history.append({"epoch": epoch, "loss": avg_loss, "roc_auc": auc, "pr_auc": prauc})
        print(f"{name} epoch {epoch:02d}/{epochs} | loss {avg_loss:.4f} | val ROC-AUC {auc:.4f} | PR-AUC {prauc:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stalled = 0
        else:
            stalled += 1
            if stalled >= PATIENCE:
                print(f"{name} early stop at epoch {epoch} (best {best_auc:.4f} @ {best_epoch})")
                break
        if device.type == "cuda":
            torch.cuda.empty_cache()

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(x, eidx)[val_t]).cpu().numpy()
    y_true = y_all[val_idx]
    y_pred = (y_prob >= 0.5).astype(np.int32)
    cm = confusion_matrix(y_true, y_pred)
    metrics = {
        "model": name,
        "best_epoch": int(best_epoch),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }
    print(f"\n{name} best metrics:")
    for k, v in metrics.items():
        if k != "model":
            print(f"  {k}: {v}")
    print(classification_report(y_true, y_pred, target_names=["Legitimate", "Fraud"], digits=4, zero_division=0))
    return model, metrics, history, y_true, y_prob


def main():
    set_seed()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    train = pd.read_parquet(DATASET_PATH / "merged_train.parquet")
    train = train.sort_values("TransactionDT").reset_index(drop=True)
    print(f"Train rows: {len(train):,}  fraud rate: {train['isFraud'].mean():.4f}")

    feature_cols = select_node_features(list(train.columns))
    print(f"Node features ({len(feature_cols)}): {feature_cols}")

    y = train["isFraud"].to_numpy(dtype=np.int64)
    split = int(len(train) * 0.8)
    train_idx = np.arange(0, split)
    val_idx = np.arange(split, len(train))
    print(f"Temporal split  train={len(train_idx):,}  valid={len(val_idx):,}")
    print(f"Fraud rate train={y[train_idx].mean():.4f}  valid={y[val_idx].mean():.4f}")

    scaler = StandardScaler()
    X = train[feature_cols].to_numpy(dtype=np.float32)
    X[train_idx] = scaler.fit_transform(X[train_idx])
    X[val_idx] = scaler.transform(X[val_idx])
    x_all = torch.from_numpy(X)

    print("Building importance-stitched temporal k-NN edges...", flush=True)
    edge_keys = resolve_edge_keys(train)
    print("Edge keys:", flush=True)
    for spec in edge_keys:
        print(f"  {spec['source']:10s}  {spec['name']}  <-  {' + '.join(spec['parts'])}", flush=True)
    edge_index, edge_stats = build_union_edges(train, edge_keys, k=K_NEIGHBORS)
    print(f"union unique directed edges: {edge_index.shape[1]:,}", flush=True)

    indptr, csr_indices = to_csr(edge_index, len(train))
    deg = np.diff(indptr)
    print(
        f"avg degree {deg.mean():.2f}  max {deg.max()}  isolated {(deg == 0).sum():,}"
    )
    edge_index_t = torch.from_numpy(edge_index.astype(np.int64))
    del edge_index, indptr, csr_indices
    gc.collect()

    n_pos = max(int(y[train_idx].sum()), 1)
    n_neg = len(train_idx) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)
    print(f"pos_weight={float(pos_weight):.2f}")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    all_metrics = []
    histories = {}

    sage = GraphSAGE(in_channels=x_all.size(1), hidden_channels=HIDDEN, dropout=DROPOUT)
    sage, sage_metrics, sage_hist, y_true, sage_prob = train_one_model(
        "GraphSAGE",
        sage,
        x_all,
        y,
        train_idx,
        val_idx,
        edge_index_t,
        pos_weight,
        device,
        EPOCHS_SAGE,
    )
    all_metrics.append(sage_metrics)
    histories["GraphSAGE"] = sage_hist
    torch.save(sage.state_dict(), RESULTS_DIR / f"graphsage_{timestamp}.pt")
    sage.cpu()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

    del sage
    gc.collect()

    gat_prob = None
    gat = GATNet(in_channels=x_all.size(1), hidden_channels=32, heads=2, dropout=DROPOUT)
    try:
        gat, gat_metrics, gat_hist, _, gat_prob = train_one_model(
            "GAT",
            gat,
            x_all,
            y,
            train_idx,
            val_idx,
            edge_index_t,
            pos_weight,
            device,
            EPOCHS_GAT,
        )
        all_metrics.append(gat_metrics)
        histories["GAT"] = gat_hist
        torch.save(gat.state_dict(), RESULTS_DIR / f"gat_{timestamp}.pt")
    except RuntimeError as exc:
        if not _is_oom(exc):
            raise
        print(
            f"GAT OOM on {device} with {edge_index_t.shape[1]:,} edges "
            f"({exc}). Skipping GAT; GraphSAGE weights are already saved."
        )
        gat_prob = None
    finally:
        del gat
        gc.collect()

    metadata = {
        "timestamp": timestamp,
        "device": str(device),
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "num_nodes": int(len(train)),
        "num_features": int(x_all.size(1)),
        "feature_names": feature_cols,
        "k_neighbors": K_NEIGHBORS,
        "edge_keys": edge_keys,
        "edge_stats": edge_stats,
        "graph_construction": "importance-stitched identity keys",
        "hidden": HIDDEN,
        "training_mode": "full-batch",
        "num_neighbors": NUM_NEIGHBORS,
        "pos_weight": float(pos_weight),
        "temporal_split": 0.8,
        "gat_trained": gat_prob is not None,
    }
    return _write_run_artifacts(
        timestamp,
        all_metrics,
        histories,
        y_true,
        sage_prob,
        gat_prob,
        metadata,
    )

def _is_oom(exc: BaseException) -> bool:
    msg = str(exc).lower()
    return "out of memory" in msg or "not enough memory" in msg


def _write_run_artifacts(
    timestamp: str,
    all_metrics: list,
    histories: dict,
    y_true,
    sage_prob,
    gat_prob,
    metadata: dict,
) -> pd.DataFrame:
    metrics_df = pd.DataFrame(all_metrics)
    metrics_df.to_parquet(RESULTS_DIR / f"gnn_results_{timestamp}.parquet", index=False)
    metrics_df.to_csv(RESULTS_DIR / f"gnn_results_{timestamp}.csv", index=False)

    pred = {"y_true": y_true, "graphsage_prob": sage_prob}
    if gat_prob is not None:
        pred["gat_prob"] = gat_prob
    pd.DataFrame(pred).to_parquet(RESULTS_DIR / f"gnn_val_predictions_{timestamp}.parquet", index=False)

    comparison_rows = list(all_metrics)
    ml_path = SAVED_PATH / "ml_results.parquet"
    if ml_path.exists():
        tab = pd.read_parquet(ml_path)
        tab = tab[~tab["Model"].str.contains("SMOTE|Undersampling|Original", regex=True)]
        for model_name in [
            "RF - Baseline",
            "RF - Remove V",
            "RF - Remove id + V",
            "LightGBM - Baseline",
            "LightGBM - Remove V",
            "LightGBM - Remove id + V",
            "XGBoost - Baseline",
            "XGBoost - Remove V",
            "XGBoost - Remove id + V",
        ]:
            hit = tab[tab["Model"] == model_name]
            if hit.empty:
                continue
            row = hit.iloc[0]
            comparison_rows.append(
                {
                    "model": row["Model"],
                    "best_epoch": None,
                    "accuracy": float(row["Accuracy"]),
                    "precision": float(row["Precision"]),
                    "recall": float(row["Recall"]),
                    "f1": float(row["F1"]),
                    "roc_auc": float(row["ROC-AUC"]),
                    "pr_auc": float(row["PR-AUC"]),
                    "balanced_accuracy": float(row["Balanced Accuracy"]),
                    "mcc": float(row["MCC"]),
                    "tn": int(row["TN"]) if "TN" in row else None,
                    "fp": int(row["FP"]) if "FP" in row else None,
                    "fn": int(row["FN"]) if "FN" in row else None,
                    "tp": int(row["TP"]) if "TP" in row else None,
                }
            )
    comparison = pd.DataFrame(comparison_rows)
    comparison.to_parquet(RESULTS_DIR / f"gnn_vs_tabular_{timestamp}.parquet", index=False)
    comparison.to_csv(RESULTS_DIR / f"gnn_vs_tabular_{timestamp}.csv", index=False)
    print("\nComparison:")
    print(comparison[["model", "roc_auc", "pr_auc", "f1", "recall"]].to_string(index=False))

    metadata = {**metadata, "metrics": all_metrics, "history": histories}
    with open(RESULTS_DIR / f"gnn_metadata_{timestamp}.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    print(f"\nSaved artifacts under {RESULTS_DIR}")
    return metrics_df


ModuleNotFoundError: No module named 'torch'

In [ ]:
ml = pd.read_parquet(SAVED_PATH / "ml_results.parquet")
ml = ml.loc[~ml["Model"].str.contains("SMOTE|Undersampling|Original", regex=True)].copy()


def group_drop(df, prefix):
    base = df[df["Model"] == f"{prefix} - Baseline"].iloc[0]
    rows = []
    for g in ["C", "D", "M", "id", "V"]:
        hit = df[df["Model"] == f"{prefix} - Remove {g}"]
        if hit.empty:
            continue
        rows.append(
            {
                "model": prefix,
                "removed": g,
                "delta_roc_auc": float(base["ROC-AUC"] - hit.iloc[0]["ROC-AUC"]),
                "delta_pr_auc": float(base["PR-AUC"] - hit.iloc[0]["PR-AUC"]),
                "removed_roc_auc": float(hit.iloc[0]["ROC-AUC"]),
            }
        )
    return pd.DataFrame(rows)


ablation = pd.concat(
    [
        group_drop(ml, "RF"),
        group_drop(ml, "LightGBM"),
        group_drop(ml, "XGBoost"),
    ],
    ignore_index=True,
)
print("ROC-AUC drop when a feature group is removed (positive = group was useful)")
display(ablation.pivot(index="removed", columns="model", values="delta_roc_auc").sort_values("LightGBM", ascending=False))

imp_path = SAVED_PATH / "lgb_column_importance.parquet"
if imp_path.exists():
    imp = pd.read_parquet(imp_path)
    print("\nTop 20 LightGBM gain features on the Remove-id+V set")
    display(imp.head(20))
    print("Gain by group")
    display(imp.groupby("group")["gain_importance"].sum().sort_values(ascending=False).to_frame())
else:
    print("saved/lgb_column_importance.parquet not found")


## 2.1 Graph construction

- **Nodes:** one per transaction in `merged_train.parquet` (590,540).
- **Node features:** the compact tabular set from `02_ml_models.ipynb` — everything except `isFraud`, `TransactionID`, `id_*`, and `V*`. Identity columns stay as features *and* define edges.
- **Edges:** for each stitched identity key (single pieces + concatenated pairs + `uid`/`uid2`), sort the group by `TransactionDT` and link every transaction to its **K = 5** nearest neighbors (undirected). Skip missing sentinels (`-999`, `-1`) and groups larger than 5% of the data. Union and deduplicate.
- **Split:** sort by `TransactionDT`, first 80% train / last 20% validation — same protocol as the RF/LGB/XGB notebooks.
- **Imbalance:** `BCEWithLogitsLoss` with `pos_weight = n_neg / n_pos` (~27.6), not accuracy.

Example stitch: `link_card1_addr1` connects two transactions only if they share both `card1` *and* `addr1`. That is a tighter chain than `card1` alone, and a different cut than `uid2`.


In [ ]:
import torch

print(f"PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

train = pd.read_parquet(DATASET_PATH / "merged_train.parquet")
train = train.sort_values("TransactionDT").reset_index(drop=True)
feature_cols = select_node_features(list(train.columns))
print(f"Rows: {len(train):,}  fraud rate: {train['isFraud'].mean():.4f}")
print(f"Node features ({len(feature_cols)}):")
print(feature_cols)


In [ ]:
# Rank identity pieces from the saved RF / LightGBM / XGBoost models, then stitch pairs.
try:
    importance = load_model_importances()
    edge_keys = propose_edge_keys(train, importance)
    save_schema(importance, edge_keys)
    display(
        importance.head(20)[
            ["feature", "RandomForest", "LightGBM", "XGBoost", "mean", "rank", "is_identity_piece"]
        ]
    )
    print("Identity pieces (consensus rank)")
    display(importance.loc[importance["is_identity_piece"], ["feature", "mean", "rank"]])
except Exception as exc:
    print(f"Could not load pickles ({exc}); using saved schema or uid/uid2 fallback")
    edge_keys = resolve_edge_keys(train)

print("Stitched edge keys")
pd.DataFrame(edge_keys)

In [ ]:
K = 5
edge_index, edge_stats = build_union_edges(train, edge_keys, k=K)
indptr, indices = to_csr(edge_index, len(train))
deg = np.diff(indptr)

print(pd.DataFrame(edge_stats)[["name", "source", "directed_edges"]].to_string(index=False))
print(f"union unique edges : {edge_index.shape[1]:,}")
print(f"average degree     : {deg.mean():.2f}")
print(f"max degree         : {deg.max()}")
print(f"isolated nodes     : {(deg == 0).sum():,}")
print(f"GraphSAGE in={len(feature_cols)} hidden=64  |  GAT in={len(feature_cols)} hidden=32 heads=2")
print(GraphSAGE(len(feature_cols)))
print(GATNet(len(feature_cols)))

## 2.2 Models

**GraphSAGE** (Hamilton et al.): each layer combines a node with the *mean* of its neighbors, then a linear map. Two layers, hidden 64, dropout 0.3, full-graph (one pass per epoch).

**GAT** (Veličković et al.): full-graph attention over the same stitched edges (2 heads → 1 head). Slightly slower; useful if some neighbors should dominate. The uid/uid2-only run: GraphSAGE 0.77 ROC-AUC after 25 epochs (still climbing); GAT 0.73. Tabular Remove-id+V models remain ~0.91.

This run uses the importance-stitched identity graph defined above (single pieces + concatenated pairs, plus uid/uid2).

Message passing is pure PyTorch `index_add_` (no `torch-sparse`).


In [ ]:
metrics = main()
display(metrics)


## 3. Train SIGN GraphSAGE (v2)

Beats the 2026-08-20 GraphSAGE holdout (**ROC-AUC 0.767**, TP 3143, TN 67984, **FP 46060**).

This run uses **SIGN-style GraphSAGE**: past-only identity edges, then 1-hop and 2-hop *mean* neighbor features concatenated with the node itself, scored by an MLP. That is the same mean-aggregation as GraphSAGE, precomputed so CPU training finishes and we can verify TP/TN.

Also: milder `sqrt(n_neg/n_pos)` class weight, and an **MCC threshold** so true positives and true negatives are both high.

Kernel: `graph`. Holdout comparison is in `03_graph_models_result.ipynb`.


In [1]:
import gc
import json
import random
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from torch import nn

warnings.filterwarnings("ignore")

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

ROOT = Path("/content/drive/MyDrive/minor-thesis") if IS_COLAB else Path.cwd()
DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
RESULTS_DIR = SAVED_PATH / "graph_models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
K_NEIGHBORS = 8
HIDDEN = 128
DROPOUT = 0.2
EPOCHS = 40
PATIENCE = 8
BATCH = 8192
LR = 1e-3
MISSING_VALUES = {-999, -1}
MAX_GROUP_FRAC = 0.05
PREV = {"roc_auc": 0.7667, "tp": 3143, "tn": 67984, "fp": 46060, "fn": 921}

EDGE_KEYS = [
    {"name": "uid", "parts": ["uid"], "source": "engineered"},
    {"name": "uid2", "parts": ["uid2"], "source": "engineered"},
    {"name": "card1", "parts": ["card1"], "source": "single"},
    {"name": "link_card1_card2", "parts": ["card1", "card2"], "source": "pair"},
    {"name": "link_card1_addr1", "parts": ["card1", "addr1"], "source": "pair"},
    {"name": "link_uid_addr1", "parts": ["uid", "addr1"], "source": "pair"},
    {"name": "link_card1_R_emaildomain", "parts": ["card1", "R_emaildomain"], "source": "pair"},
]


def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def select_node_features(columns):
    return [
        c
        for c in columns
        if c not in {"isFraud", "TransactionID"}
        and not c.startswith("id")
        and not c.startswith("V")
    ]


def key_series(df, parts):
    if len(parts) == 1:
        return df[parts[0]]
    cols = [df[p].to_numpy() for p in parts]
    return pd.Series(list(zip(*cols)), index=df.index)


def build_past_knn(df, parts, k=K_NEIGHBORS):
    keys = key_series(df, parts)
    invalid = pd.Series(False, index=df.index)
    for p in parts:
        invalid |= df[p].isin(MISSING_VALUES)
    rows, cols = [], []
    max_group = int(len(df) * MAX_GROUP_FRAC)
    work = df.loc[~invalid, ["TransactionDT"]].copy()
    work["_k"] = keys.loc[~invalid].to_numpy()
    for _, group in work.groupby("_k", sort=False):
        n = len(group)
        if n < 2 or n > max_group:
            continue
        idx = group.sort_values("TransactionDT").index.to_numpy(dtype=np.int64, copy=False)
        for offset in range(1, min(k + 1, n)):
            earlier, later = idx[:-offset], idx[offset:]
            rows.append(later)
            cols.append(earlier)
    if not rows:
        return np.zeros((2, 0), dtype=np.int64)
    return np.vstack([np.concatenate(rows), np.concatenate(cols)])


def coalesce_edges(edge_index, num_nodes):
    if edge_index.size == 0:
        return edge_index
    packed = edge_index[0].astype(np.int64) * np.int64(num_nodes) + edge_index[1].astype(np.int64)
    _, uniq = np.unique(packed, return_index=True)
    return edge_index[:, np.sort(uniq)]


def build_union_edges(df, edge_keys, k=K_NEIGHBORS):
    stats, chunks = [], []
    for spec in edge_keys:
        parts = [p for p in spec["parts"] if p in df.columns]
        if len(parts) != len(spec["parts"]):
            print(f"  skip {spec['name']}", flush=True)
            continue
        edges = build_past_knn(df, parts, k=k)
        stats.append({**spec, "directed_edges": int(edges.shape[1])})
        print(f"  {spec['name']:28s}  edges={edges.shape[1]:,}", flush=True)
        if edges.shape[1]:
            chunks.append(edges)
    union = (
        coalesce_edges(np.concatenate(chunks, axis=1), len(df))
        if chunks
        else np.zeros((2, 0), dtype=np.int64)
    )
    return union, stats


def sign_features(X, edge_index, num_nodes):
    """Precompute 1-hop and 2-hop mean neighbor features (SIGN / simplified SAGE)."""
    if edge_index.size == 0:
        zeros = np.zeros_like(X)
        deg = np.zeros((num_nodes, 1), dtype=np.float32)
        return np.hstack([X, zeros, zeros, deg])
    row, col = edge_index[0], edge_index[1]
    A = csr_matrix(
        (np.ones(row.shape[0], dtype=np.float32), (row, col)),
        shape=(num_nodes, num_nodes),
    )
    deg = np.asarray(A.sum(axis=1), dtype=np.float32).ravel()
    inv = np.reciprocal(np.clip(deg, 1.0, None))
    hop1 = inv[:, None] * (A @ X)
    hop2 = inv[:, None] * (A @ hop1)
    return np.hstack([X, hop1.astype(np.float32), hop2.astype(np.float32), np.log1p(deg)[:, None]])


class SignMLP(nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_channels, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def metrics_dict(name, y_true, y_prob, threshold, best_epoch=None):
    y_pred = (y_prob >= threshold).astype(np.int32)
    cm = confusion_matrix(y_true, y_pred)
    row = {
        "model": name,
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }
    if best_epoch is not None:
        row["best_epoch"] = int(best_epoch)
    return row


def tune_threshold(y_true, y_prob, min_recall=0.30):
    best_mcc, best_thr = -1.0, 0.5
    for thr in np.linspace(0.05, 0.95, 91):
        pred = (y_prob >= thr).astype(np.int32)
        if recall_score(y_true, pred, zero_division=0) < min_recall:
            continue
        mcc = matthews_corrcoef(y_true, pred)
        if mcc > best_mcc:
            best_mcc, best_thr = mcc, float(thr)
    return best_thr


set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, flush=True)

train = pd.read_parquet(DATASET_PATH / "merged_train.parquet")
train = train.sort_values("TransactionDT").reset_index(drop=True)
print(f"rows={len(train):,} fraud={train['isFraud'].mean():.4f}", flush=True)

feature_cols = select_node_features(list(train.columns))
y = train["isFraud"].to_numpy(dtype=np.int64)
split = int(len(train) * 0.8)
train_idx = np.arange(0, split)
val_idx = np.arange(split, len(train))

scaler = StandardScaler()
X = train[feature_cols].to_numpy(dtype=np.float32)
X[train_idx] = scaler.fit_transform(X[train_idx])
X[val_idx] = scaler.transform(X[val_idx])

print("Building past-only identity edges...", flush=True)
edge_index, edge_stats = build_union_edges(train, EDGE_KEYS, k=K_NEIGHBORS)
print(f"union directed edges: {edge_index.shape[1]:,}", flush=True)
deg = np.bincount(edge_index[0], minlength=len(train)) if edge_index.size else np.zeros(len(train))
print(f"avg in-degree {deg.mean():.2f}  isolated {(deg == 0).sum():,}", flush=True)

print("Precomputing 1-hop / 2-hop mean features...", flush=True)
Z = sign_features(X, edge_index, len(train))
print(f"SIGN feature dim={Z.shape[1]}", flush=True)
del X, edge_index
gc.collect()

n_pos = max(int(y[train_idx].sum()), 1)
n_neg = len(train_idx) - n_pos
pos_weight = torch.tensor([float(np.sqrt(n_neg / n_pos))], dtype=torch.float32)
print(f"pos_weight={float(pos_weight):.2f}", flush=True)

z_all = torch.from_numpy(Z)
y_t = torch.from_numpy(y.astype(np.float32))
model = SignMLP(Z.shape[1]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

history = []
best_auc, best_epoch, stalled = -1.0, 0, 0
best_state = None
rng = np.random.default_rng(RANDOM_SEED)

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = rng.permutation(train_idx)
    total = 0.0
    nbat = 0
    for start in range(0, len(perm), BATCH):
        b = perm[start : start + BATCH]
        xb = z_all[b].to(device)
        yb = y_t[b].to(device)
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        total += float(loss.item())
        nbat += 1
    sched.step()
    model.eval()
    with torch.no_grad():
        logits = []
        for start in range(0, len(val_idx), BATCH):
            b = val_idx[start : start + BATCH]
            logits.append(model(z_all[b].to(device)).cpu())
        y_prob = torch.sigmoid(torch.cat(logits)).numpy()
    auc = roc_auc_score(y[val_idx], y_prob)
    prauc = average_precision_score(y[val_idx], y_prob)
    avg = total / max(nbat, 1)
    history.append({"epoch": epoch, "loss": avg, "roc_auc": float(auc), "pr_auc": float(prauc)})
    print(f"epoch {epoch:02d}/{EPOCHS} loss {avg:.4f} ROC-AUC {auc:.4f} PR-AUC {prauc:.4f}", flush=True)
    if auc > best_auc + 1e-4:
        best_auc, best_epoch, stalled = auc, epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        stalled += 1
        if stalled >= PATIENCE:
            print(f"early stop at {epoch} best {best_auc:.4f} @{best_epoch}", flush=True)
            break

if best_state is not None:
    model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    logits = []
    for start in range(0, len(val_idx), BATCH):
        b = val_idx[start : start + BATCH]
        logits.append(model(z_all[b].to(device)).cpu())
    y_prob = torch.sigmoid(torch.cat(logits)).numpy()
y_true = y[val_idx]

thr = tune_threshold(y_true, y_prob, min_recall=0.30)
at_half = metrics_dict("GraphSAGE-SIGN-v2 @0.5", y_true, y_prob, 0.5, best_epoch)
at_mcc = metrics_dict("GraphSAGE-SIGN-v2 MCC-thr", y_true, y_prob, thr, best_epoch)

print(f"\nChosen MCC threshold: {thr:.2f}")
print(f"Previous GraphSAGE: ROC-AUC {PREV['roc_auc']:.4f} TP {PREV['tp']} TN {PREV['tn']} FP {PREV['fp']}")
for row in (at_half, at_mcc):
    print(f"\n{row['model']}")
    for k in ["threshold", "roc_auc", "pr_auc", "f1", "mcc", "recall", "precision", "tn", "fp", "fn", "tp"]:
        print(f"  {k}: {row[k]}")
    print(
        classification_report(
            y_true,
            (y_prob >= row["threshold"]).astype(int),
            target_names=["Legitimate", "Fraud"],
            digits=4,
            zero_division=0,
        )
    )

print(f"ROC-AUC delta: {at_mcc['roc_auc'] - PREV['roc_auc']:+.4f}")
print(f"TP delta: {at_mcc['tp'] - PREV['tp']:+d}  TN delta: {at_mcc['tn'] - PREV['tn']:+d}")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
torch.save(
    {"state_dict": model.state_dict(), "in_channels": Z.shape[1], "feature_cols": feature_cols},
    RESULTS_DIR / f"graphsage_v2_{timestamp}.pt",
)
payload = {
    "timestamp": timestamp,
    "variant": "GraphSAGE-SIGN-v2",
    "k_neighbors": K_NEIGHBORS,
    "hidden": HIDDEN,
    "edge_keys": EDGE_KEYS,
    "edge_stats": edge_stats,
    "pos_weight": float(pos_weight),
    "best_epoch": int(best_epoch),
    "threshold_mcc": float(thr),
    "previous": PREV,
    "metrics_0.5": at_half,
    "metrics_mcc_threshold": at_mcc,
    "history": history,
    "node_features": feature_cols,
}
with open(RESULTS_DIR / f"gnn_v2_metadata_{timestamp}.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)
pd.DataFrame([at_half, at_mcc]).to_csv(RESULTS_DIR / f"gnn_v2_results_{timestamp}.csv", index=False)
print("saved", RESULTS_DIR / f"graphsage_v2_{timestamp}.pt", flush=True)

Device: cpu
rows=590,540 fraud=0.0350
Building past-only identity edges...
  uid                           edges=4,396,626
  uid2                          edges=2,159,334
  card1                         edges=4,398,625
  link_card1_card2              edges=4,328,822
  link_card1_addr1              edges=3,454,801
  link_uid_addr1                edges=3,441,563
  link_card1_R_emaildomain      edges=4,183,074
union directed edges: 8,408,088
avg in-degree 14.24  isolated 13,088
Precomputing 1-hop / 2-hop mean features...
SIGN feature dim=187
pos_weight=5.24
epoch 01/40 loss 0.4499 ROC-AUC 0.7957 PR-AUC 0.2326
epoch 02/40 loss 0.3816 ROC-AUC 0.8165 PR-AUC 0.2798
epoch 03/40 loss 0.3623 ROC-AUC 0.8258 PR-AUC 0.2963
epoch 04/40 loss 0.3518 ROC-AUC 0.8283 PR-AUC 0.3047
epoch 05/40 loss 0.3422 ROC-AUC 0.8321 PR-AUC 0.3038
epoch 06/40 loss 0.3356 ROC-AUC 0.8338 PR-AUC 0.3178
epoch 07/40 loss 0.3293 ROC-AUC 0.8369 PR-AUC 0.3252
epoch 08/40 loss 0.3240 ROC-AUC 0.8329 PR-AUC 0.3155
epoch 09/40 los